# Complete Analysis Workflow

This tutorial demonstrates a complete neuroimaging analysis workflow using all three main nltools classes.

## Load Example Data

In [ ]:
from nltools.datasets import fetch_pain
from nltools.data import Brain_Data, Design_Matrix
from nltools.utils import get_resource_path
import pandas as pd
import numpy as np
from scipy.stats import ttest_ind

# Load example data
data = fetch_pain()
brain = data[:28]  # Use first 28 subjects
print(f"Loaded {len(brain)} subjects")

## Metadata and Group Analysis

Let's create some simulated metadata and perform a group comparison:

In [ ]:
# Create simulated pain ratings
np.random.seed(42)
pain_ratings = np.random.normal(5, 2, size=len(brain))
pain_ratings = np.clip(pain_ratings, 1, 10)  # Keep ratings between 1-10

# Create metadata
brain.X = pd.DataFrame({
    'SubjectID': [f'sub-{i:02d}' for i in range(len(brain))],
    'PainRating': pain_ratings,
    'Group': ['high' if rating > 5 else 'low' for rating in pain_ratings]
})

print(brain.X.head())
print(f"\nHigh pain group: {sum(brain.X['Group'] == 'high')} subjects")
print(f"Low pain group: {sum(brain.X['Group'] == 'low')} subjects")

## Group Comparison

In [ ]:
# Split into groups
high_group = brain[brain.X['Group'] == 'high']
low_group = brain[brain.X['Group'] == 'low']

# Calculate mean activation for each group
high_mean = high_group.mean()
low_mean = low_group.mean()

# Calculate difference
difference = high_mean - low_mean

# Plot the difference
difference.plot(limit=5)

## Statistical Testing

In [ ]:
# One-sample t-test on the difference
t_results = difference.ttest(threshold_dict={'fdr': 0.05})

# Plot thresholded results
if 'thr_t' in t_results:
    t_results['thr_t'].plot(limit=5)
    print(f"Found {np.sum(t_results['thr_t'].data != 0)} significant voxels at FDR < 0.05")
else:
    print("No significant voxels found at FDR < 0.05")

## Regression Analysis

Let's perform a regression analysis using continuous pain ratings:

In [ ]:
# Create design matrix for regression
brain.X = pd.DataFrame({
    'intercept': 1,
    'pain_rating': pain_ratings
})

# Run regression
regression_results = brain.regress()

# Plot beta map for pain rating
regression_results['beta'][1].plot(limit=5)

## Region of Interest Analysis

In [ ]:
# Threshold the t-statistic map to create an ROI
roi = regression_results['t'][1].threshold(upper='95%', binarize=True)

# Extract mean activity from ROI
roi_activity = brain.extract_roi(roi)

# Plot relationship between ROI activity and pain ratings
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 6))
plt.scatter(pain_ratings, roi_activity)
plt.xlabel('Pain Rating')
plt.ylabel('Mean ROI Activity')
plt.title('Relationship between Pain Rating and Brain Activity')

# Calculate correlation
from scipy.stats import pearsonr
r, p = pearsonr(pain_ratings, roi_activity)
plt.text(0.1, 0.9, f'r = {r:.3f}, p = {p:.3f}', transform=plt.gca().transAxes)
plt.show()

## Summary

In this tutorial, we demonstrated:
- Loading neuroimaging data
- Performing group comparisons
- Statistical testing with multiple comparison correction
- Regression analysis
- ROI extraction and analysis

This workflow can be adapted for your own analyses by substituting your data and experimental design.